# APP Assignment: Solutions 1 to 65

Run the cells from top to bottom. Every problem has its own markdown note followed by one code cell. Inputs that the assignment reads from the user are replaced with sample values so the whole notebook runs without stopping for typing.

## 1. Single Number

XOR of a number with itself is 0, and XOR with 0 leaves the number unchanged. So XOR-ing the whole array cancels every pair and leaves the single element. One pass, no extra space.

In [1]:
from functools import reduce
from operator import xor

def single_number(nums):
    return reduce(xor, nums)

for nums in ([2, 2, 1], [4, 1, 2, 1, 2], [1]):
    print(nums, "->", single_number(nums))

[2, 2, 1] -> 1
[4, 1, 2, 1, 2] -> 4
[1] -> 1


## 2. Find Palindromes in a Given Range

Convert each number to a string and compare it with its reverse.

In [2]:
def palindromes_in_range(m, n):
    return [x for x in range(m, n + 1) if str(x) == str(x)[::-1]]

for m, n in ((10, 50), (100, 150), (1, 9)):
    print(m, n, "->", palindromes_in_range(m, n))

10 50 -> [11, 22, 33, 44]
100 150 -> [101, 111, 121, 131, 141]
1 9 -> [1, 2, 3, 4, 5, 6, 7, 8, 9]


## 3. Regular Expression Matching

Bottom-up DP. `dp[i][j]` tells whether `s[i:]` matches `p[j:]`. When the next pattern character is `*`, either skip the pair (zero copies) or consume one character of `s` and stay on the same pattern position.

In [3]:
def is_match(s, p):
    m, n = len(s), len(p)
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[m][n] = True
    for i in range(m, -1, -1):
        for j in range(n - 1, -1, -1):
            first = i < m and p[j] in (s[i], ".")
            if j + 1 < n and p[j + 1] == "*":
                dp[i][j] = dp[i][j + 2] or (first and dp[i + 1][j])
            else:
                dp[i][j] = first and dp[i + 1][j + 1]
    return dp[0][0]

for s, p in (("abcd", ".*"), ("mississippi", "misisp*."), ("aa", "a")):
    print(repr(s), repr(p), "->", is_match(s, p))

'abcd' '.*' -> True
'mississippi' 'misisp*.' -> False
'aa' 'a' -> False


## 4. Decode Ways with Pattern Generation

Backtracking: at each position take one digit (if it is not `0`) or two digits (if they form 10 to 26). Every path that reaches the end of the string is one decoded word.

In [4]:
def decode_ways(s):
    results = []

    def go(i, path):
        if i == len(s):
            results.append("".join(path))
            return
        if s[i] == "0":
            return
        path.append(chr(64 + int(s[i])))
        go(i + 1, path)
        path.pop()
        if i + 1 < len(s) and 10 <= int(s[i:i + 2]) <= 26:
            path.append(chr(64 + int(s[i:i + 2])))
            go(i + 2, path)
            path.pop()

    go(0, [])
    return results

for s in ("12", "226", "11106", "06"):
    res = decode_ways(s)
    print("Input:", s)
    print("Total Ways:", len(res))
    print("Decoded Strings:")
    print("\n".join(res) if res else "None")
    print()

Input: 12
Total Ways: 2
Decoded Strings:
AB
L

Input: 226
Total Ways: 3
Decoded Strings:
BBF
BZ
VF

Input: 11106
Total Ways: 2
Decoded Strings:
AAJF
KJF

Input: 06
Total Ways: 0
Decoded Strings:
None



### Tree helpers (used by problems 5, 10 and 27)

`build_tree` turns a level-order list (with `None` for missing nodes) into linked nodes. `tree_to_list` goes the other way.

In [5]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def tree_to_list(root):
    out = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

## 5. Balanced Binary Tree

A helper returns the height of a subtree, or -1 as soon as any subtree is unbalanced. That keeps the check at O(n).

In [6]:
def is_balanced(root):
    def height(node):
        if node is None:
            return 0
        left = height(node.left)
        if left == -1:
            return -1
        right = height(node.right)
        if right == -1 or abs(left - right) > 1:
            return -1
        return max(left, right) + 1
    return height(root) != -1

for values in ([3, 9, 20, None, None, 15, 7], [1, 2, 2, 3, 3, None, None, 4, 4], [4]):
    print(values, "->", is_balanced(build_tree(values)))

[3, 9, 20, None, None, 15, 7] -> True
[1, 2, 2, 3, 3, None, None, 4, 4] -> False
[4] -> True


## 6. Reverse Integer

Digits are peeled off one at a time. Before each step the code checks whether appending the next digit would pass the 32-bit limit, so no value larger than 32 bits is ever built. The limit for negatives is 2^31, for positives 2^31 - 1.

In [7]:
def reverse_integer(x):
    sign = -1 if x < 0 else 1
    limit = 2 ** 31 if x < 0 else 2 ** 31 - 1
    x = abs(x)
    rev = 0
    while x:
        digit = x % 10
        x //= 10
        if rev > limit // 10 or (rev == limit // 10 and digit > limit % 10):
            return 0
        rev = rev * 10 + digit
    return sign * rev

for x in (123, -123, 120, 1534236469, -2147483648):
    print(x, "->", reverse_integer(x))

123 -> 321
-123 -> -321
120 -> 21
1534236469 -> 0
-2147483648 -> 0


## 7. Minimize Maximum Consecutive Zeros with Construction

All strings with exactly K ones come from choosing the K positions of the ones. The longest zero block is the longest piece after splitting on `1`.

In [8]:
from itertools import combinations

def max_zero_block(s):
    return max(len(part) for part in s.split("1"))

def minimize_zero_blocks(L, K):
    strings = []
    for positions in combinations(range(L), K):
        chars = ["0"] * L
        for p in positions:
            chars[p] = "1"
        strings.append("".join(chars))
    strings.sort(reverse=True)
    scores = {s: max_zero_block(s) for s in strings}
    best = min(scores.values())
    optimal = [s for s in strings if scores[s] == best]
    print("Possible Strings:")
    for s in strings:
        print(f"{s} -> max zeros = {scores[s]}")
    print("Optimal Arrangement(s):", ", ".join(optimal))
    print("Minimum Longest Zero Block:", best)

for L, K in ((4, 2), (5, 0), (5, 5)):
    print(f"Input: {L} {K}")
    minimize_zero_blocks(L, K)
    print()

Input: 4 2
Possible Strings:
1100 -> max zeros = 2
1010 -> max zeros = 1
1001 -> max zeros = 2
0110 -> max zeros = 1
0101 -> max zeros = 1
0011 -> max zeros = 2
Optimal Arrangement(s): 1010, 0110, 0101
Minimum Longest Zero Block: 1

Input: 5 0
Possible Strings:
00000 -> max zeros = 5
Optimal Arrangement(s): 00000
Minimum Longest Zero Block: 5

Input: 5 5
Possible Strings:
11111 -> max zeros = 0
Optimal Arrangement(s): 11111
Minimum Longest Zero Block: 0



## 8. Merge Triplets to Form Target Triplet

A triplet is only useful if none of its values exceed the target. Take the maximum of every useful triplet position by position and check whether that equals the target.

In [9]:
def merge_triplets(triplets, target):
    best = [0, 0, 0]
    for t in triplets:
        if all(t[i] <= target[i] for i in range(3)):
            for i in range(3):
                best[i] = max(best[i], t[i])
    return best == target

tests = [
    ([[2, 5, 3], [1, 8, 4], [1, 7, 5]], [2, 7, 5]),
    ([[3, 4, 5], [4, 5, 6]], [3, 2, 5]),
    ([[2, 5, 3], [2, 3, 4], [1, 2, 5], [5, 2, 3]], [5, 5, 5]),
]
for triplets, target in tests:
    print(target, "->", merge_triplets(triplets, target))

[2, 7, 5] -> True
[3, 2, 5] -> False
[5, 5, 5] -> True


## 9. Palindromic Substrings with Pattern Listing

Expand around each of the 2n-1 centres while the two ends match. Results are sorted by length and then by starting position, which gives the order used in the examples.

In [10]:
def palindromic_substrings(s):
    found = []
    n = len(s)
    for center in range(2 * n - 1):
        left = center // 2
        right = left + center % 2
        while left >= 0 and right < n and s[left] == s[right]:
            found.append((right - left + 1, left, s[left:right + 1]))
            left -= 1
            right += 1
    found.sort()
    return [item[2] for item in found]

for s in ("ababa", "racecar", "level"):
    subs = palindromic_substrings(s)
    print("Input:", s)
    print("Total Count:", len(subs))
    print("Palindromic Substrings:")
    print("\n".join(subs))
    print()

Input: ababa
Total Count: 9
Palindromic Substrings:
a
b
a
b
a
aba
bab
aba
ababa

Input: racecar
Total Count: 10
Palindromic Substrings:
r
a
c
e
c
a
r
cec
aceca
racecar

Input: level
Total Count: 7
Palindromic Substrings:
l
e
v
e
l
eve
level



## 10. Kth Smallest Element in a BST

An in-order walk of a BST visits values in ascending order, so stop at the kth visit.

In [11]:
def kth_smallest(root, k):
    stack = []
    node = root
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        k -= 1
        if k == 0:
            return node.val
        node = node.right

print(kth_smallest(build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13]), 5))
print(kth_smallest(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3))

7
3


## 11. Missing Number

The sum of 0..n is n(n+1)/2. Subtract the sum of the array and what is left is the missing number.

In [12]:
def missing_number(nums):
    n = len(nums)
    return n * (n + 1) // 2 - sum(nums)

for nums in ([3, 0, 1], [0, 1], [9, 6, 4, 2, 3, 5, 7, 0, 1]):
    print(nums, "->", missing_number(nums))

[3, 0, 1] -> 2
[0, 1] -> 2
[9, 6, 4, 2, 3, 5, 7, 0, 1] -> 8


## 12. Coin Change

`dp[a]` is the fewest coins needed for amount `a`. Each amount tries every coin and keeps the best.

In [13]:
def coin_change(coins, amount):
    INF = float("inf")
    dp = [0] + [INF] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and dp[a - c] + 1 < dp[a]:
                dp[a] = dp[a - c] + 1
    return -1 if dp[amount] == INF else dp[amount]

for coins, amount in (([1, 2, 5], 11), ([2], 3), ([1], 0)):
    print(coins, amount, "->", coin_change(coins, amount))

[1, 2, 5] 11 -> 3
[2] 3 -> -1
[1] 0 -> 0


## 13. Cheapest Flights Within K Stops

Bellman-Ford limited to k+1 rounds. Each round extends every best-known path by one more flight, using the prices from the previous round only.

In [14]:
def find_cheapest_price(n, flights, src, dst, k):
    INF = float("inf")
    prices = [INF] * n
    prices[src] = 0
    for _ in range(k + 1):
        nxt = prices[:]
        for u, v, w in flights:
            if prices[u] != INF and prices[u] + w < nxt[v]:
                nxt[v] = prices[u] + w
        prices = nxt
    return -1 if prices[dst] == INF else prices[dst]

print(find_cheapest_price(4, [[0, 1, 100], [1, 2, 100], [2, 0, 100], [1, 3, 600], [2, 3, 200]], 0, 3, 1))
print(find_cheapest_price(3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 1))
print(find_cheapest_price(3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 0))

700
200
500


### Linked list helpers (used by problem 14)

In [15]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build_list(values):
    dummy = tail = ListNode()
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def list_to_py(node):
    out = []
    while node:
        out.append(node.val)
        node = node.next
    return out

## 14. Merge k Sorted Lists

A min-heap holds the current head of every list. Pop the smallest, attach it to the result, and push that list's next node.

In [16]:
import heapq

def merge_k_lists(lists):
    heap = []
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))
    dummy = tail = ListNode()
    while heap:
        val, i, node = heapq.heappop(heap)
        tail.next = node
        tail = node
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next

for data in ([[1, 4, 5], [1, 3, 4], [2, 6]], [[4, 5], [1, 6]]):
    merged = merge_k_lists([build_list(v) for v in data])
    print(data, "->", list_to_py(merged))

[[1, 4, 5], [1, 3, 4], [2, 6]] -> [1, 1, 2, 3, 4, 4, 5, 6]
[[4, 5], [1, 6]] -> [1, 4, 5, 6]


## 15. Reverse Bits

Shift the answer left and drop in the lowest bit of `n`, 32 times.

In [17]:
def reverse_bits(n):
    result = 0
    for _ in range(32):
        result = (result << 1) | (n & 1)
        n >>= 1
    return result

for n in (43261596, 2147483644):
    print(n, "->", reverse_bits(n))

43261596 -> 964176192
2147483644 -> 1073741822


## 16. Separate Letters and Numbers from a String

In [18]:
def separate(s):
    letters = "".join(ch for ch in s if ch.isalpha())
    digits = "".join(ch for ch in s if ch.isdigit())
    return [letters, digits]

for s in ("a1b2c3d4", "x9y8z7", "12345", "abcdef"):
    print(s, "->", separate(s))

a1b2c3d4 -> ['abcd', '1234']
x9y8z7 -> ['xyz', '987']
12345 -> ['', '12345']
abcdef -> ['abcdef', '']


## 17. Merge Intervals

Sort by start. If the next interval starts at or before the end of the last merged one, extend the end; otherwise begin a new interval.

In [19]:
def merge_intervals(intervals):
    merged = []
    for start, end in sorted(intervals):
        if merged and start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

for iv in ([[1, 3], [2, 6], [8, 10], [15, 18]], [[1, 4], [4, 5]], [[4, 7], [1, 4]]):
    print(iv, "->", merge_intervals(iv))

[[1, 3], [2, 6], [8, 10], [15, 18]] -> [[1, 6], [8, 10], [15, 18]]
[[1, 4], [4, 5]] -> [[1, 5]]
[[4, 7], [1, 4]] -> [[1, 7]]


## 18. Maximum Product Subarray

Track both the maximum and the minimum product ending at each position, since a negative number can turn the smallest product into the largest.

In [20]:
def max_product(nums):
    best = cur_max = cur_min = nums[0]
    for x in nums[1:]:
        candidates = (x, cur_max * x, cur_min * x)
        cur_max, cur_min = max(candidates), min(candidates)
        best = max(best, cur_max)
    return best

for nums in ([2, 3, -2, 4], [-2, 0, -1]):
    print(nums, "->", max_product(nums))

[2, 3, -2, 4] -> 6
[-2, 0, -1] -> 0


## 19. Minimum Team Selection to Cover Required Skills

Each candidate becomes a bitmask over the required skills. A dictionary maps every skill-set reached so far to the smallest team that produces it. After all candidates are processed, the entry for the full mask is the answer.

In [21]:
def min_team(required, candidates):
    index = {skill: i for i, skill in enumerate(required)}
    masks = [sum(1 << index[s] for s in c if s in index) for c in candidates]
    full = (1 << len(required)) - 1
    best = {0: []}
    for i, m in enumerate(masks):
        for mask, team in list(best.items()):
            new = mask | m
            if new not in best or len(best[new]) > len(team) + 1:
                best[new] = team + [i]
    return best[full]

cases = [
    (["Python", "SQL", "MachineLearning", "DataStructures"],
     [["Python", "SQL"], ["SQL", "MachineLearning"], ["MachineLearning", "DataStructures"], ["DataStructures"]]),
    (["Java", "Python", "SQL"], [["Java"], ["Python", "SQL"], ["SQL"]]),
    (["C++", "Python", "DataStructures"], [["C++"], ["Python"], ["DataStructures"], ["C++", "Python", "DataStructures"]]),
]
for required, candidates in cases:
    print(*min_team(required, candidates))

0 2
0 1
3


## 20. Word Break with All Segmentations

Recursive search from each start index with memoisation, building every sentence that uses dictionary words only.

Note on Example 1 in the assignment: with `["Amrita", "Rita", "Mysuru"]` the string `AmritaMysuru` splits as `Amrita Mysuru`. The sample output in the PDF shows an extra `rita`, which does not match the input, so the code prints the correct segmentation.

In [22]:
from functools import lru_cache

def word_break(s, word_dict):
    words = set(word_dict)

    @lru_cache(maxsize=None)
    def build(start):
        if start == len(s):
            return [[]]
        out = []
        for end in range(start + 1, len(s) + 1):
            word = s[start:end]
            if word in words:
                for rest in build(end):
                    out.append([word] + rest)
        return out

    return [" ".join(p) for p in build(0)]

tests = [
    ("AmritaMysuru", ["Amrita", "Rita", "Mysuru"]),
    ("applepenapple", ["apple", "pen"]),
    ("catsanddog", ["cat", "cats", "and", "sand", "dog"]),
    ("catsandog", ["cats", "dog", "sand", "and", "cat"]),
]
for s, d in tests:
    result = word_break(s, d)
    print("Can Segment:", bool(result))
    print("Segmentations:")
    print("\n".join(result) if result else "None")
    print()

Can Segment: True
Segmentations:
Amrita Mysuru

Can Segment: True
Segmentations:
apple pen apple

Can Segment: True
Segmentations:
cat sand dog
cats and dog

Can Segment: False
Segmentations:
None



## 21. Number of 1 Bits

In [23]:
def hamming_weight(n):
    count = 0
    while n:
        n &= n - 1
        count += 1
    return count

for n in (11, 128, 2147483645):
    print(n, "->", hamming_weight(n))

11 -> 3
128 -> 1
2147483645 -> 30


## 22. Midpoint of Line Segment

In [24]:
def midpoint(p1, p2):
    mx = (p1[0] + p2[0]) / 2
    my = (p1[1] + p2[1]) / 2
    fix = lambda v: int(v) if v == int(v) else v
    return (fix(mx), fix(my))

for p1, p2 in (((0, 0), (4, 4)), ((1, 3), (5, 7)), ((-2, 4), (2, 0))):
    print(p1, p2, "->", midpoint(p1, p2))

(0, 0) (4, 4) -> (2, 2)
(1, 3) (5, 7) -> (3, 5)
(-2, 4) (2, 0) -> (0, 2)


## 23. Sum of Cubes in a Given Range

The sum of the first n cubes is (n(n+1)/2)^2, so the range sum is the difference of two such values.

In [25]:
def sum_of_cubes(a, b):
    f = lambda n: (n * (n + 1) // 2) ** 2
    return f(b) - f(a - 1)

for a, b in ((4, 9), (1, 3), (5, 5)):
    print(a, b, "->", sum_of_cubes(a, b))

4 9 -> 1989
1 3 -> 36
5 5 -> 125


## 24. Closest Pair of Points

Sort by x. For each point, only look at later points whose x-distance is still smaller than the best distance found, and stop as soon as it is not.

In [26]:
import math

def closest_pair(points):
    pts = sorted(points)
    best = float("inf")
    for i in range(len(pts)):
        for j in range(i + 1, len(pts)):
            dx = pts[j][0] - pts[i][0]
            if dx * dx >= best:
                break
            dy = pts[j][1] - pts[i][1]
            best = min(best, dx * dx + dy * dy)
    return math.sqrt(best)

tests = [[(0, 0), (1, 1)], [(1, 1), (2, 2), (5, 5)], [(0, 0), (3, 4), (6, 8)]]
for pts in tests:
    print(pts, "->", round(closest_pair(pts), 4))

[(0, 0), (1, 1)] -> 1.4142
[(1, 1), (2, 2), (5, 5)] -> 1.4142
[(0, 0), (3, 4), (6, 8)] -> 5.0


## 25. Count Trailing Zeros in Binary

`n & -n` isolates the lowest set bit. Its bit length minus one is the number of trailing zeros.

In [27]:
def trailing_zeros(n):
    return (n & -n).bit_length() - 1

for n in (8, 12):
    print(n, bin(n), "->", trailing_zeros(n))

8 0b1000 -> 3
12 0b1100 -> 2


## 26. Sum of Unique Elements

In [28]:
from collections import Counter

def sum_unique(arr):
    counts = Counter(arr)
    unique = [x for x in arr if counts[x] == 1]
    print("Unique Elements:", *unique if unique else ["None"])
    print("Sum:", sum(unique))

for arr in ([1, 3, 2, 3, 2, 4], [10, 20, 10, 30, 40, 30], [5, 5, 5, 5]):
    sum_unique(arr)
    print()

Unique Elements: 1 4
Sum: 5

Unique Elements: 20 40
Sum: 60

Unique Elements: None
Sum: 0



## 27. Invert Binary Tree

In [29]:
def invert_tree(root):
    if root:
        root.left, root.right = invert_tree(root.right), invert_tree(root.left)
    return root

for values in ([4, 2, 7, 1, 3, 6, 9], [2, 1, 3]):
    print(values, "->", tree_to_list(invert_tree(build_tree(values))))

[4, 2, 7, 1, 3, 6, 9] -> [4, 7, 2, 9, 6, 3, 1]
[2, 1, 3] -> [2, 3, 1]


## 28. Valid Anagram

In [30]:
from collections import Counter

def is_anagram(s, t):
    return Counter(s) == Counter(t)

for s, t in (("anagram", "nagaram"), ("rat", "car")):
    print(s, t, "->", is_anagram(s, t))

anagram nagaram -> True
rat car -> False


## 29. Happy Number

Keep replacing the number by the sum of the squares of its digits. If a value repeats before reaching 1, there is a cycle and the number is not happy.

In [31]:
def is_happy(n):
    seen = set()
    while n != 1 and n not in seen:
        seen.add(n)
        n = sum(int(d) ** 2 for d in str(n))
    return n == 1

for n in (19, 2):
    print(n, "->", is_happy(n))

19 -> True
2 -> False


## 30. Set Matrix Zeroes

Record which rows and columns contain a zero, then zero them directly in the original matrix.

In [32]:
def set_zeroes(matrix):
    zero_rows = {i for i, row in enumerate(matrix) if 0 in row}
    zero_cols = {j for row in matrix for j, v in enumerate(row) if v == 0}
    for i in range(len(matrix)):
        for j in range(len(matrix[0])):
            if i in zero_rows or j in zero_cols:
                matrix[i][j] = 0

for matrix in ([[1, 1, 1], [1, 0, 1], [1, 1, 1]], [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]):
    set_zeroes(matrix)
    print(matrix)

[[1, 0, 1], [0, 0, 0], [1, 0, 1]]
[[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]]


## 31. Card Probabilities

Count aces, face cards (J, Q, K) and red cards (hearts and diamonds). Black is everything that is not red.

Note on the assignment's sample: it lists only AH, 10D, 2H and JD as red, but KH is a heart and is red too. That makes 5 red cards out of 8, so the code prints P(Red)=0.625, most frequent category Red, and P(Black)=0.375.

In [33]:
def card_probabilities(cards):
    total = len(cards)
    ace = sum(1 for c in cards if c[:-1] == "A")
    face = sum(1 for c in cards if c[:-1] in ("J", "Q", "K"))
    red = sum(1 for c in cards if c[-1] in ("H", "D"))
    counts = {"Ace": ace, "Face": face, "Red": red}
    top = max(counts, key=counts.get)
    print(f"P(Ace)={ace / total:.3f} P(Face)={face / total:.3f} P(Red)={red / total:.3f}")
    print("Most Frequent Category:", top)
    print(f"P(Black)={(total - red) / total:.3f}")

card_probabilities(["AH", "10D", "KC", "QS", "2H", "JD", "7C", "KH"])

P(Ace)=0.125 P(Face)=0.500 P(Red)=0.625
Most Frequent Category: Red
P(Black)=0.375


## 32. Toggle K-th Bit

XOR with `1 << k` flips bit k (counting from 0 at the right).

In [34]:
def toggle_bit(n, k):
    return n ^ (1 << k)

for n, k in ((5, 1), (4, 2)):
    print(n, k, "->", toggle_bit(n, k))

5 1 -> 7
4 2 -> 0


## 33. Plus One

Walk from the last digit. A 9 becomes 0 and the carry moves left. The first non-9 digit gets incremented and the work is done. If every digit was 9, put a 1 in front.

In [35]:
def plus_one(digits):
    for i in range(len(digits) - 1, -1, -1):
        if digits[i] < 9:
            digits[i] += 1
            return digits
        digits[i] = 0
    return [1] + digits

for d in ([1, 2, 3], [4, 3, 2, 1], [9]):
    print(d, "->", plus_one(d[:]))

[1, 2, 3] -> [1, 2, 4]
[4, 3, 2, 1] -> [4, 3, 2, 2]
[9] -> [1, 0]


## 34. String Transformation with Given Substring

Find the substring, shift each of its letters one step back (with `a` wrapping to `z`) and stitch the string back together.

In [36]:
def transform(s, sub):
    i = s.find(sub)
    shifted = "".join(chr((ord(c) - 97 - 1) % 26 + 97) for c in sub)
    return s[:i] + shifted + s[i + len(sub):]

for s, sub in (("hackerrank", "hack"), ("programming", "gram"), ("apple", "a")):
    print(s, sub, "->", transform(s, sub))

hackerrank hack -> gzbjerrank
programming gram -> profqzlming
apple a -> zpple


## 35. Trapping Rain Water

Two pointers move inward from both ends. The side with the smaller wall decides the water level at its position, because the other side is at least as tall.

In [37]:
def trap(height):
    left, right = 0, len(height) - 1
    left_max = right_max = water = 0
    while left < right:
        if height[left] < height[right]:
            left_max = max(left_max, height[left])
            water += left_max - height[left]
            left += 1
        else:
            right_max = max(right_max, height[right])
            water += right_max - height[right]
            right -= 1
    return water

for h in ([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1], [4, 2, 0, 3, 2, 5]):
    print(h, "->", trap(h))

[0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1] -> 6
[4, 2, 0, 3, 2, 5] -> 9


## 36. Generate Parentheses

Add an opening bracket while fewer than n are used. Add a closing bracket only when it would not exceed the number of openings.

In [38]:
def generate_parentheses(n):
    result = []

    def build(current, opened, closed):
        if len(current) == 2 * n:
            result.append(current)
            return
        if opened < n:
            build(current + "(", opened + 1, closed)
        if closed < opened:
            build(current + ")", opened, closed + 1)

    build("", 0, 0)
    return result

for n in (3, 1):
    print(n, "->", generate_parentheses(n))

3 -> ['((()))', '(()())', '(())()', '()(())', '()()()']
1 -> ['()']


## 37. Spiral Matrix

Take the top row, then rotate the remaining matrix counter-clockwise so the next edge to read becomes the new top row.

In [39]:
def spiral_order(matrix):
    rows = [row[:] for row in matrix]
    result = []
    while rows:
        result += rows.pop(0)
        rows = [list(r) for r in zip(*rows)][::-1]
    return result

print(spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))
print(spiral_order([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]))

[1, 2, 3, 6, 9, 8, 7, 4, 5]
[1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]


## 38. Container With Most Water

Start with the widest container and always move the pointer at the shorter line, since moving the taller one can only shrink the area.

In [40]:
def max_area(height):
    left, right = 0, len(height) - 1
    best = 0
    while left < right:
        best = max(best, (right - left) * min(height[left], height[right]))
        if height[left] < height[right]:
            left += 1
        else:
            right -= 1
    return best

for h in ([1, 8, 6, 2, 5, 4, 8, 3, 7], [1, 1], [4, 3, 2, 1, 4]):
    print(h, "->", max_area(h))

[1, 8, 6, 2, 5, 4, 8, 3, 7] -> 49
[1, 1] -> 1
[4, 3, 2, 1, 4] -> 16


## 39. Matrix Chain Multiplication

`dp[i][j]` is the cheapest cost for multiplying matrices i to j. Every split point k is tried, and the best k is stored so the parenthesisation and the multiplication steps can be rebuilt.

In [41]:
def matrix_chain(dims):
    n = len(dims) - 1
    dp = [[0] * n for _ in range(n)]
    split = [[0] * n for _ in range(n)]
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            dp[i][j] = float("inf")
            for k in range(i, j):
                cost = dp[i][k] + dp[k + 1][j] + dims[i] * dims[k + 1] * dims[j + 1]
                if cost < dp[i][j]:
                    dp[i][j] = cost
                    split[i][j] = k
    steps = []

    def build(i, j):
        if i == j:
            return f"A{i + 1}"
        k = split[i][j]
        left = build(i, k)
        right = build(k + 1, j)
        expr = f"({left} × {right})"
        steps.append((expr, dims[i] * dims[k + 1] * dims[j + 1]))
        return expr

    expr = build(0, n - 1)
    return dp[0][n - 1], expr, steps

for dims in ([10, 30, 5, 60], [40, 20, 30, 10, 30]):
    total, expr, steps = matrix_chain(dims)
    print("Dimensions:", dims)
    print("Minimum Cost:", total)
    print("Optimal Parenthesization:", expr)
    for i, (e, c) in enumerate(steps, 1):
        print(f"Step {i}: {e} -> cost = {c}")
    print("Total Cost:", " + ".join(str(c) for _, c in steps), "=", total)
    print()

Dimensions: [10, 30, 5, 60]
Minimum Cost: 4500
Optimal Parenthesization: ((A1 × A2) × A3)
Step 1: (A1 × A2) -> cost = 1500
Step 2: ((A1 × A2) × A3) -> cost = 3000
Total Cost: 1500 + 3000 = 4500

Dimensions: [40, 20, 30, 10, 30]
Minimum Cost: 26000
Optimal Parenthesization: ((A1 × (A2 × A3)) × A4)
Step 1: (A2 × A3) -> cost = 6000
Step 2: (A1 × (A2 × A3)) -> cost = 8000
Step 3: ((A1 × (A2 × A3)) × A4) -> cost = 12000
Total Cost: 6000 + 8000 + 12000 = 26000



## 40. Validate and Complete a 9×9 Sudoku Board

Validation checks sub-grids, rows and columns for repeated digits and reports the first problem found. A valid board is then solved with backtracking: try each digit in an empty cell, recurse, and undo if it leads to a dead end.

In [42]:
def validate_sudoku(board):
    for b in range(9):
        r0, c0 = 3 * (b // 3), 3 * (b % 3)
        cells = [board[r0 + i][c0 + j] for i in range(3) for j in range(3)]
        seen = set()
        for v in cells:
            if v != "." and v in seen:
                names = ["top-left", "top-middle", "top-right", "middle-left", "center",
                         "middle-right", "bottom-left", "bottom-middle", "bottom-right"]
                return False, f"Duplicate '{v}' found in the {names[b]} 3×3 sub-grid"
            seen.add(v)
    for r in range(9):
        seen = set()
        for v in board[r]:
            if v != "." and v in seen:
                return False, f"Duplicate '{v}' found in row {r + 1}"
            seen.add(v)
    for c in range(9):
        seen = set()
        for r in range(9):
            v = board[r][c]
            if v != "." and v in seen:
                return False, f"Duplicate '{v}' found in column {c + 1}"
            seen.add(v)
    return True, ""

def solve_sudoku(board):
    for r in range(9):
        for c in range(9):
            if board[r][c] == ".":
                for d in "123456789":
                    ok = all(board[r][j] != d for j in range(9)) and \
                         all(board[i][c] != d for i in range(9)) and \
                         all(board[3 * (r // 3) + i][3 * (c // 3) + j] != d
                             for i in range(3) for j in range(3))
                    if ok:
                        board[r][c] = d
                        if solve_sudoku(board):
                            return True
                        board[r][c] = "."
                return False
    return True

def print_board(board):
    for r in range(9):
        if r in (3, 6):
            print("-" * 27)
        print(" | ".join(" ".join(board[r][c:c + 3]) for c in (0, 3, 6)))

valid_board = [list(row) for row in [
    "53..7....", "6..195...", ".98....6.", "8...6...3", "4..8.3..1",
    "7...2...6", ".6....28.", "...419..5", "....8..79"]]
invalid_board = [list(row) for row in [
    "83..7....", "6..195...", ".98....6.", "8...6...3", "4..8.3..1",
    "7...2...6", ".6....28.", "...419..5", "....8..79"]]

for board in (valid_board, invalid_board):
    ok, reason = validate_sudoku(board)
    print("Valid Sudoku:", str(ok).lower())
    if ok:
        solve_sudoku(board)
        print("Completed Sudoku:")
        print_board(board)
    else:
        print("Reason:", reason)
    print()

Valid Sudoku: true
Completed Sudoku:
5 3 4 | 6 7 8 | 9 1 2
6 7 2 | 1 9 5 | 3 4 8
1 9 8 | 3 4 2 | 5 6 7
---------------------------
8 5 9 | 7 6 1 | 4 2 3
4 2 6 | 8 5 3 | 7 9 1
7 1 3 | 9 2 4 | 8 5 6
---------------------------
9 6 1 | 5 3 7 | 2 8 4
2 8 7 | 4 1 9 | 6 3 5
3 4 5 | 2 8 6 | 1 7 9

Valid Sudoku: false
Reason: Duplicate '8' found in the top-left 3×3 sub-grid



## 41. Find All Words in Grid

Words go into a trie. A DFS walks the board and follows the trie at the same time, so a path is abandoned as soon as no word starts with the letters seen so far. Visited cells are marked with `#` and restored on the way back.

In [43]:
def find_words(board, words):
    trie = {}
    for w in words:
        node = trie
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = w
    m, n = len(board), len(board[0])
    found = set()

    def dfs(r, c, node):
        ch = board[r][c]
        if ch not in node:
            return
        nxt = node[ch]
        if "$" in nxt:
            found.add(nxt["$"])
        board[r][c] = "#"
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and board[nr][nc] != "#":
                dfs(nr, nc, nxt)
        board[r][c] = ch

    for r in range(m):
        for c in range(n):
            dfs(r, c, trie)
    return sorted(found)

grid1 = [["o", "a", "a", "n"], ["e", "t", "a", "e"], ["i", "h", "k", "r"], ["i", "f", "l", "v"]]
print(find_words(grid1, ["oath", "pea", "eat", "rain"]))
print(find_words([["a", "b"], ["c", "d"]], ["abcb"]))

['eat', 'oath']
[]


## 42. House Robber

At each house the best total is either the best up to the previous house (skip this one) or the best up to two houses back plus this one.

In [44]:
def rob(nums):
    prev, cur = 0, 0
    for x in nums:
        prev, cur = cur, max(cur, prev + x)
    return cur

for nums in ([1, 2, 3, 1], [2, 7, 9, 3, 1]):
    print(nums, "->", rob(nums))

[1, 2, 3, 1] -> 4
[2, 7, 9, 3, 1] -> 12


## 43. Find Euler Path

The graph needs all its edges in one connected piece and either 0 or 2 odd-degree vertices. The path itself comes from Hierholzer's algorithm: keep walking along unused edges, and when stuck, back out and record the vertex.

In [45]:
def euler_path(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    for a in adj:
        a.sort()
    degree = [len(a) for a in adj]
    odd = [i for i in range(n) if degree[i] % 2]
    with_edges = [i for i in range(n) if degree[i]]

    connected = True
    if with_edges:
        seen = {with_edges[0]}
        stack = [with_edges[0]]
        while stack:
            v = stack.pop()
            for u in adj[v]:
                if u not in seen:
                    seen.add(u)
                    stack.append(u)
        connected = all(i in seen for i in with_edges)

    exists = connected and len(odd) in (0, 2)
    print("Euler Path Exists:", str(exists).lower())
    print("Node Degrees:")
    for i in range(n):
        print(f"{i} -> {degree[i]}")
    print("Odd Degree Nodes:", odd)

    if not exists:
        reasons = []
        if len(odd) > 2:
            reasons.append("More than 2 nodes have odd degree")
        if not connected:
            reasons.append("graph is disconnected")
        print("Reason:", " and ".join(reasons))
        return None

    start = odd[0] if odd else (with_edges[0] if with_edges else 0)
    print("Start Node:", start)
    work = [a[:] for a in adj]
    stack, path = [start], []
    while stack:
        v = stack[-1]
        if work[v]:
            u = work[v].pop(0)
            work[u].remove(v)
            stack.append(u)
        else:
            path.append(stack.pop())
    path.reverse()
    print("Euler Path:", " → ".join(map(str, path)))
    return path

tests = [
    (4, [[0, 1], [1, 2], [2, 0], [1, 3]]),
    (4, [[0, 1], [1, 2], [2, 3]]),
    (4, [[0, 1], [2, 3]]),
]
for n, edges in tests:
    euler_path(n, edges)
    print()

Euler Path Exists: true
Node Degrees:
0 -> 2
1 -> 3
2 -> 2
3 -> 1
Odd Degree Nodes: [1, 3]
Start Node: 1
Euler Path: 1 → 0 → 2 → 1 → 3

Euler Path Exists: true
Node Degrees:
0 -> 1
1 -> 2
2 -> 2
3 -> 1
Odd Degree Nodes: [0, 3]
Start Node: 0
Euler Path: 0 → 1 → 2 → 3

Euler Path Exists: false
Node Degrees:
0 -> 1
1 -> 1
2 -> 1
3 -> 1
Odd Degree Nodes: [0, 1, 2, 3]
Reason: More than 2 nodes have odd degree and graph is disconnected



## 44. Rotate Image

Transpose the matrix in place, then reverse every row. Together these give a 90 degree clockwise rotation.

In [46]:
def rotate(matrix):
    n = len(matrix)
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
    for row in matrix:
        row.reverse()

for matrix in ([[1, 2, 3], [4, 5, 6], [7, 8, 9]],
               [[5, 1, 9, 11], [2, 4, 8, 10], [13, 3, 6, 7], [15, 14, 12, 16]]):
    rotate(matrix)
    print(matrix)

[[7, 4, 1], [8, 5, 2], [9, 6, 3]]
[[15, 13, 2, 5], [14, 3, 4, 1], [12, 6, 8, 9], [16, 7, 10, 11]]


## 45. Climbing Stairs with Patterns

The count follows the Fibonacci recurrence. Listing every pattern is done by recursion. For large n the number of patterns is far too big to print, so the listing is limited to n up to 15.

In [47]:
def total_ways(n):
    a, b = 1, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b

def climb_patterns(n):
    patterns = []

    def build(remaining, steps):
        if remaining == 0:
            patterns.append(" + ".join(map(str, steps)))
            return
        for step in (1, 2):
            if step <= remaining:
                build(remaining - step, steps + [step])

    build(n, [])
    return patterns

for n in (2, 3, 4, 45):
    print("Input: n =", n)
    print("Total Ways:", total_ways(n))
    if n <= 15:
        print("Patterns:")
        print("\n".join(climb_patterns(n)))
    else:
        print("Patterns: too many to list")
    print()

Input: n = 2
Total Ways: 2
Patterns:
1 + 1
2

Input: n = 3
Total Ways: 3
Patterns:
1 + 1 + 1
1 + 2
2 + 1

Input: n = 4
Total Ways: 5
Patterns:
1 + 1 + 1 + 1
1 + 1 + 2
1 + 2 + 1
2 + 1 + 1
2 + 2

Input: n = 45
Total Ways: 1836311903
Patterns: too many to list



## 46. Filter Characters from a String

In [48]:
def filter_characters(target, reference):
    remove = set(reference)
    return "".join(ch for ch in target if ch not in remove)

for target, reference in (("programming", "grm"), ("hello world", "lo"), ("abcdef", "xyz")):
    print(repr(filter_characters(target, reference)))

'poain'
'he wrd'
'abcdef'


## 47. Sum of Odd Numbers in a List

Three versions: a function that takes the list as a parameter and loops, a list comprehension, and a user-defined helper `is_odd` used inside another function.

In [49]:
def sum_odd_parameter(nums):
    total = 0
    for x in nums:
        if x % 2 != 0:
            total += x
    return total

def sum_odd_comprehension(nums):
    return sum([x for x in nums if x % 2 != 0])

def is_odd(x):
    return x % 2 != 0

def sum_odd_user_defined(nums):
    return sum(x for x in nums if is_odd(x))

for nums in ([1, 2, 3, 4, 5, 6], [10, 15, 20, 25, 30, 35], [2, 4, 6, 8]):
    print(nums, sum_odd_parameter(nums), sum_odd_comprehension(nums), sum_odd_user_defined(nums))

[1, 2, 3, 4, 5, 6] 9 9 9
[10, 15, 20, 25, 30, 35] 75 75 75
[2, 4, 6, 8] 0 0 0


## 48. Palindrome Number Using a Decorator Function

The decorator wraps the checker and prints the sentence for the result, so the checking function itself stays untouched.

In [50]:
def palindrome_message(func):
    def wrapper(num):
        result = func(num)
        if result:
            print(f"{num} is a Palindrome")
        else:
            print(f"{num} is Not a Palindrome")
        return result
    return wrapper

@palindrome_message
def is_palindrome(num):
    return str(num) == str(num)[::-1]

for num in (121, 1234):
    is_palindrome(num)

121 is a Palindrome
1234 is Not a Palindrome


### Input helper (used by problems 49, 53, 54, 59 and 60)

`make_input` returns a function that behaves like `input()` but reads from a prepared list and echoes the prompt with the value. To type values yourself, replace `ask` with `input` in the cell that uses it.

In [51]:
def make_input(values):
    it = iter(values)

    def ask(prompt=""):
        value = next(it)
        print(prompt + str(value))
        return value

    return ask

## 49. Cumulative Average Using a Closure

The outer function owns the list. The inner function appends to it and returns the running average, and the list survives between calls because the inner function holds a reference to it.

In [52]:
def cumulative_average():
    numbers = []

    def add(x):
        numbers.append(x)
        return sum(numbers) / len(numbers)

    return add

ask = make_input([4, 10, 20, 30, 40])
n = int(ask("Enter n: "))
average = cumulative_average()
for _ in range(n):
    value = int(ask("Enter number: "))
    print("Cumulative Average:", average(value))

Enter n: 4
Enter number: 10
Cumulative Average: 10.0
Enter number: 20
Cumulative Average: 15.0
Enter number: 30
Cumulative Average: 20.0
Enter number: 40
Cumulative Average: 25.0


## 50. Remove Elements Greater Than 20 Using Decorator and filter()

In [53]:
def show_result(func):
    def wrapper(nums):
        result = func(nums)
        print("Filtered list:", result)
        return result
    return wrapper

@show_result
def remove_greater_than_20(nums):
    return list(filter(lambda x: x <= 20, nums))

remove_greater_than_20([5, 10, 23, 40, 8])

Filtered list: [5, 10, 8]


[5, 10, 8]

## 51. File and Directory Management Using os and time

In [54]:
import os
import time

folder = os.path.join("My_Project", "assignments", "task1")
os.makedirs(folder, exist_ok=True)

print("Current Working Directory:", os.getcwd())
print("Byte String Format:", os.getcwdb())

file_path = os.path.join(folder, "sample.py")
with open(file_path, "w") as f:
    f.write("print('Python Assignment')")

print(os.path.isdir(folder))

total_size = 0
for root, dirs, files in os.walk("My_Project"):
    for name in files:
        total_size += os.path.getsize(os.path.join(root, name))
print("total_size:", total_size, "bytes")

print("access time:", time.ctime(os.path.getatime(file_path)))
print("Last modified time:", time.ctime(os.path.getmtime(file_path)))
print("list of files in the directory:", os.listdir(folder))

Current Working Directory: c:\Degree\MCA\Sem 3\Python\Assignments
Byte String Format: b'c:\\Degree\\MCA\\Sem 3\\Python\\Assignments'
True
total_size: 26 bytes
access time: Sat Nov  7 16:06:47 2026
Last modified time: Sat Nov  7 16:06:47 2026
list of files in the directory: ['sample.py']


## 52. CSV Data Cleaning and JSON Conversion

The PDF shows the CSV on one line because the line breaks were lost in formatting, so the sample is written here as a proper multiline string. Whitespace is stripped from every field, and a record is skipped when age or salary is missing, marked `Unknown`, or fails conversion.

In [55]:
import csv
import json

raw_data = """Name,Age,Salary,Join Date
Alice, 30, 95000, 2015-03-01
Bob, , 60000, 2018-07-15
Charlie, 35, Unknown, 2024-02-01
Diana, 40, 110000, 2021-11-03"""

lines = raw_data.split("\n")
reader = csv.DictReader(lines)

cleaned_records = []
for row in reader:
    row = {k.strip(): (v or "").strip() for k, v in row.items()}
    if not row["Age"] or not row["Salary"] or row["Salary"] == "Unknown":
        continue
    try:
        age = int(row["Age"])
        salary = float(row["Salary"])
    except ValueError:
        continue
    cleaned_records.append({
        "name": row["Name"],
        "age": age,
        "salary": salary,
        "join_date": row["Join Date"],
    })

with open("cleaned_data.json", "w") as f:
    json.dump(cleaned_records, f, indent=2)

with open("cleaned_data.json") as f:
    print(f.read())

[
  {
    "name": "Alice",
    "age": 30,
    "salary": 95000.0,
    "join_date": "2015-03-01"
  },
  {
    "name": "Diana",
    "age": 40,
    "salary": 110000.0,
    "join_date": "2021-11-03"
  }
]


## 53. Longest Substring Without Repeating Characters

A boolean array of 26 entries tracks which letters are inside the current window. On a repeat, the left edge moves forward, clearing letters, until the duplicate is gone.

In [56]:
def longest_unique_substring(s):
    visited = [False] * 26
    left = best = 0
    for right, ch in enumerate(s):
        idx = ord(ch) - 97
        while visited[idx]:
            visited[ord(s[left]) - 97] = False
            left += 1
        visited[idx] = True
        best = max(best, right - left + 1)
    return best

ask = make_input(["abcabcbb", "bbbbb", "pwwkew"])
for _ in range(3):
    s = ask("Enter string: ")
    print("Length of the longest unique substring:", longest_unique_substring(s))

Enter string: abcabcbb
Length of the longest unique substring: 3
Enter string: bbbbb
Length of the longest unique substring: 1
Enter string: pwwkew
Length of the longest unique substring: 3


## 54. Generator Function to Extract Vowels

In [57]:
def getvowels(name):
    vowels = "aeiouAEIOU"
    for ch in name:
        if ch in vowels:
            yield ch

ask = make_input(["Priyanka"])
name = ask("Enter name: ")
for v in getvowels(name):
    print(v)

Enter name: Priyanka
i
a
a


## 55. User-Defined Iterator for Decimal Values

`__iter__` returns the object itself. `__next__` hands back the current value and moves ahead by `step`, raising `StopIteration` once the value passes `stop`. The tiny tolerance guards against floating point drift for steps like 0.1.

In [58]:
class NumberIterator:
    def __init__(self, start=0.0, stop=1.0, step=0.25):
        self.current = start
        self.stop = stop
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.stop + 1e-9:
            raise StopIteration
        value = self.current
        self.current += self.step
        return value

for value in NumberIterator(0.0, 1.0, 0.25):
    print(value)

0.0
0.25
0.5
0.75
1.0


## 56. Move All Zeros to the End

In [59]:
def move_zeros(l):
    non_zero = [x for x in l if x != 0]
    zeros = [x for x in l if x == 0]
    return non_zero + zeros

for l in ([0, 1, 0, 2, 0, 3], [4, 0, 5, 0, 2, 0, 8]):
    print(l, "->", move_zeros(l))

[0, 1, 0, 2, 0, 3] -> [1, 2, 3, 0, 0, 0]
[4, 0, 5, 0, 2, 0, 8] -> [4, 5, 2, 8, 0, 0, 0]


## 57. Student Performance Analyzer

Grades are assigned from each average (A from 90, B from 80, C from 70, otherwise D), and `Counter` tallies them. `reduce` adds up every mark.

In [60]:
import random
from datetime import datetime
from collections import Counter
from functools import reduce

students = {"Tarun": [82, 91, 78], "Karthik": [65, 72, 68], "Chadrika": [91, 95, 89],
            "Trisha": [76, 81, 79], "Atul": [88, 84, 92]}

averages = {name: sum(marks) / len(marks) for name, marks in students.items()}
topper = max(averages, key=averages.get)

def grade(avg):
    if avg >= 90:
        return "A"
    if avg >= 80:
        return "B"
    if avg >= 70:
        return "C"
    return "D"

grade_counts = Counter(grade(avg) for avg in averages.values())
all_marks = [m for marks in students.values() for m in marks]
total_marks = reduce(lambda a, b: a + b, all_marks)

for name, avg in averages.items():
    print(f"{name}: {avg:.2f} ({grade(avg)})")
print("Highest average:", topper, f"{averages[topper]:.2f}")
print("Randomly selected student:", random.choice(list(students)))
print("Current date:", datetime.now().strftime("%d-%m-%Y"))
print("Grade distribution:", dict(grade_counts))
print("Total marks:", total_marks)

Tarun: 83.67 (B)
Karthik: 68.33 (D)
Chadrika: 91.67 (A)
Trisha: 78.67 (C)
Atul: 88.00 (B)
Highest average: Chadrika 91.67
Randomly selected student: Chadrika
Current date: 07-11-2026
Grade distribution: {'B': 2, 'D': 1, 'A': 1, 'C': 1}
Total marks: 1231


## 58. Automated Course Project Team Allocation

The team size comes from `sys.argv[1]`. Inside a notebook `sys.argv` holds Jupyter's own arguments, so the conversion falls back to 3 when it is missing or not a number. From a terminal, `python allocation.py 3` works as the assignment describes.

In [61]:
import sys
import random
from datetime import datetime
from collections import defaultdict
from itertools import combinations
from functools import reduce

students = {
    "Anita": ("CSE", 85), "Rahul": ("CSE", 78), "Priya": ("CSE", 92), "Kiran": ("CSE", 70),
    "Meena": ("Data Science", 88), "Sohan": ("Data Science", 91), "Ravi": ("Data Science", 76),
    "Divya": ("AI", 95), "Arjun": ("AI", 80), "Neha": ("AI", 84),
    "Sameer": ("Cyber Security", 89), "Pooja": ("Cyber Security", 73),
}

try:
    team_size = int(sys.argv[1])
except (IndexError, ValueError):
    team_size = 3

dept_groups = defaultdict(list)
for name, (dept, marks) in students.items():
    dept_groups[dept].append(name)

department = next((d for d in dept_groups if len(dept_groups[d]) >= team_size), None)

if department is None:
    print("No department has enough students for a team of", team_size)
else:
    possible_teams = list(combinations(dept_groups[department], team_size))
    selected_team = random.sample(possible_teams, 1)[0]
    team_marks = [students[name][1] for name in selected_team]
    total = reduce(lambda a, b: a + b, team_marks)
    average = total / team_size
    allocation_date = datetime.now().strftime("%d-%m-%Y")

    print("===== PROJECT TEAM ALLOCATION =====")
    print("Team Size:", team_size)
    print("Allocation Date:", allocation_date)
    print("Department:", department)
    print("Selected Team:")
    for i, name in enumerate(selected_team, 1):
        print(f"{i}. {name} - {students[name][1]}")
    print("Total Marks:", total)
    print(f"Average Marks: {average:.2f}")

===== PROJECT TEAM ALLOCATION =====
Team Size: 3
Allocation Date: 07-11-2026
Department: CSE
Selected Team:
1. Anita - 85
2. Priya - 92
3. Kiran - 70
Total Marks: 247
Average Marks: 82.33


## 59. Student Marks Management System

An `assert` rejects an empty name, `float()` raises `ValueError` for text like `ABC`, and the custom `InvalidMarksError` covers marks outside 0 to 100. Valid records go to `student_records.txt`. The `finally` block runs after every record, valid or not. The sample run below feeds in a mix of good and bad entries; swap `ask` for `input` to type your own.

In [62]:
class InvalidMarksError(Exception):
    pass

def process_student(name, marks_text):
    try:
        assert name.strip() != "", "Student name cannot be empty"
        marks = float(marks_text)
        if marks < 0 or marks > 100:
            raise InvalidMarksError("Marks must be between 0 and 100")
    except AssertionError as e:
        print("Error:", e)
    except ValueError:
        print("Error: Marks must be numeric")
    except InvalidMarksError as e:
        print("Error:", e)
    else:
        with open("student_records.txt", "a") as f:
            f.write(f"{name},{marks}\n")
        print("Record saved")
    finally:
        print("Record processing completed")

open("student_records.txt", "w").close()

ask = make_input(["Ravi", "85", "y", "", "70", "y", "Meena", "ABC", "y",
                  "Kiran", "150", "y", "Divya", "92", "n"])

while True:
    name = ask("Enter student name: ")
    marks_text = ask("Enter marks: ")
    process_student(name, marks_text)
    if ask("Enter another record? (y/n): ").strip().lower() != "y":
        break

print()
print("Contents of student_records.txt:")
with open("student_records.txt") as f:
    print(f.read())

Enter student name: Ravi
Enter marks: 85
Record saved
Record processing completed
Enter another record? (y/n): y
Enter student name: 
Enter marks: 70
Error: Student name cannot be empty
Record processing completed
Enter another record? (y/n): y
Enter student name: Meena
Enter marks: ABC
Error: Marks must be numeric
Record processing completed
Enter another record? (y/n): y
Enter student name: Kiran
Enter marks: 150
Error: Marks must be between 0 and 100
Record processing completed
Enter another record? (y/n): y
Enter student name: Divya
Enter marks: 92
Record saved
Record processing completed
Enter another record? (y/n): n

Contents of student_records.txt:
Ravi,85.0
Divya,92.0



## 60. Insert a Sensor Reading into a NumPy Array

`isdigit()` accepts only plain digits, so `A`, `Hello` and `25.5` are rejected with `ValueError`. A position outside 0 to len(array) raises `IndexError`. The `finally` block prints the completion message every time. Four sample attempts are run: one valid, two with bad values, one with a bad position.

In [64]:
%pip install numpy

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   --- ------------------------------------ 1.0/12.6 MB 6.0 MB/s eta 0:00:02
   ----- ---------------------------------- 1.8/12.6 MB 5.0 MB/s eta 0:00:03
   -------- ------------------------------- 2.6/12.6 MB 4.4 MB/s eta 0:00:03
   ---------- ----------------------------- 3.4/12.6 MB 4.3 MB/s eta 0:00:03
   ------------- -------------------------- 4.2/12.6 MB 4.2 MB/s eta 0:00:03
   --------------- ------------------------ 5.0/12.6 MB 4.1 MB/s eta 0:00:02
   ------------------ --------------------- 5.8/12.6 MB 4.1 MB/s eta 0:00:02
   -------------------- ------------------- 6.6/12.6 MB 4.1 MB/s eta 0:00:02
   ---------------------- ----------------- 7.1/12.6 MB 4.0 MB/s eta 0:00:02
   ----------------------- ---------------- 7.3/12.6 MB 3.8 MB/s eta 0:00:02
   ------------------------- -------------- 7.9/12.6 MB 3.6 MB/s eta 0:00:02
   --------------------------- ------------ 8.7/12.6 MB 3.6 MB/s eta 0:00:02
   ---


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [65]:
import numpy as np

def insert_reading(arr, ask):
    try:
        position_text = ask("Enter position: ")
        element_text = ask("Enter element: ")
        if not element_text.isdigit():
            raise ValueError("Only whole numbers made of digits are allowed")
        if not position_text.isdigit():
            raise ValueError("Position must be a whole number")
        position = int(position_text)
        if position > len(arr):
            raise IndexError("Invalid insertion position")
        arr = np.insert(arr, position, int(element_text))
        print("Updated array:", arr)
    except ValueError as e:
        print("ValueError:", e)
    except IndexError as e:
        print("IndexError:", e)
    finally:
        print("Insertion attempt completed")
        print()
    return arr

readings = np.array([10, 20, 30, 40, 50])
print("Original array:", readings)
print()

for pos, el in (("2", "25"), ("2", "A"), ("1", "25.5"), ("9", "25")):
    insert_reading(readings, make_input([pos, el]))

Original array: [10 20 30 40 50]

Enter position: 2
Enter element: 25
Updated array: [10 20 25 30 40 50]
Insertion attempt completed

Enter position: 2
Enter element: A
ValueError: Only whole numbers made of digits are allowed
Insertion attempt completed

Enter position: 1
Enter element: 25.5
ValueError: Only whole numbers made of digits are allowed
Insertion attempt completed

Enter position: 9
Enter element: 25
IndexError: Invalid insertion position
Insertion attempt completed



## 61. Precious Metal Loan Management

In [67]:
RATES = {
    "gold": {"18": 12.0, "22": 10.0, "24": 9.0},
    "silver": {"925": 14.0, "950": 13.0, "999": 12.0},
    "platinum": {"900": 11.0, "950": 10.0, "999": 9.0},
}

def calculate_interest(metal, purity, amount, tenure):
    metal_key = str(metal).lower()
    if metal_key not in RATES:
        raise ValueError(f"Invalid metal type: {metal}")
    purity_key = str(purity).upper().replace("K", "")
    if purity_key not in RATES[metal_key]:
        raise ValueError(f"Unsupported purity {purity} for {metal}")
    rate = RATES[metal_key][purity_key]
    return amount * rate * tenure / 100

def run_case(*args):
    try:
        interest = calculate_interest(*args)
    except TypeError as e:
        print("Missing argument:", e)
    except ValueError as e:
        print("Invalid input:", e)
    else:
        print("Interest:", interest)
    finally:
        print("Processing complete")
        print()

run_case("gold", "22K", 500000, 2)
run_case()
run_case("gold")
run_case("gold", 22)
run_case("gold", 22, 500000)
run_case("silver")
run_case("diamond", 999, 100000, 2)
run_case("gold", 15, 100000, 2)

Interest: 100000.0
Processing complete

Missing argument: calculate_interest() missing 4 required positional arguments: 'metal', 'purity', 'amount', and 'tenure'
Processing complete

Missing argument: calculate_interest() missing 3 required positional arguments: 'purity', 'amount', and 'tenure'
Processing complete

Missing argument: calculate_interest() missing 2 required positional arguments: 'amount' and 'tenure'
Processing complete

Missing argument: calculate_interest() missing 1 required positional argument: 'tenure'
Processing complete

Missing argument: calculate_interest() missing 3 required positional arguments: 'purity', 'amount', and 'tenure'
Processing complete

Invalid input: Invalid metal type: diamond
Processing complete

Invalid input: Unsupported purity 15 for gold
Processing complete



## 62. Rat in a Maze Using Backtracking

Choose a safe cell and mark it, explore down, right, up and left, and unmark the cell if none of them reach the exit.

In [68]:
def solve_maze(maze):
    n = len(maze)
    path = [[0] * n for _ in range(n)]

    def safe(r, c):
        return 0 <= r < n and 0 <= c < n and maze[r][c] == 1 and path[r][c] == 0

    def explore(r, c):
        if not safe(r, c):
            return False
        path[r][c] = 1
        if r == n - 1 and c == n - 1:
            return True
        for dr, dc in ((1, 0), (0, 1), (-1, 0), (0, -1)):
            if explore(r + dr, c + dc):
                return True
        path[r][c] = 0
        return False

    return path if explore(0, 0) else None

mazes = [
    [[1, 0, 0, 0], [1, 1, 0, 1], [0, 1, 0, 0], [1, 1, 1, 1]],
    [[1, 0, 0], [0, 0, 0], [0, 0, 1]],
]
for maze in mazes:
    result = solve_maze(maze)
    if result:
        print("Solution Path:")
        for row in result:
            print(*row)
    else:
        print("No solution exists")
    print()

Solution Path:
1 0 0 0
1 1 0 0
0 1 0 0
0 1 1 1

No solution exists



## 63. Debugging a Coin-Toss Game

Set `DEBUG = True` to make `breakpoint()` pause the game inside the notebook's debugger prompt. At the `(Pdb)` prompt type:

- `p guess`, `p toss`, `p result` to inspect values
- `p type(guess)` to inspect the type
- `n` to run the next statement
- `c` to continue

With `DEBUG = False` the game runs normally and prints the same values so you can check the logic `result == guess` without stopping.

In [69]:
import random

DEBUG = False

def play(guess):
    toss = random.choice(["heads", "tails"])
    result = "win" if toss == guess else "lose"
    if DEBUG:
        breakpoint()
    return toss, result

random.seed(3)
for guess in ("heads", "tails", "heads", "tails"):
    toss, result = play(guess)
    print(f"guess={guess} ({type(guess).__name__}) toss={toss} result={result} match={toss == guess}")

guess=heads (str) toss=heads result=win match=True
guess=tails (str) toss=heads result=lose match=False
guess=heads (str) toss=tails result=lose match=False
guess=tails (str) toss=tails result=win match=True


## 64. Printer Class and Object Identities

In [70]:
class Printer:
    def __init__(self, print_capacity, color_mode, printer_type):
        self.print_capacity = print_capacity
        self.color_mode = color_mode
        self.printer_type = printer_type

printer1 = Printer(500, "Color", "Laser")
printer2 = Printer(200, "Mono", "Inkjet")
printer3 = Printer(1000, "Mono", "Dot Matrix")

for name, p in (("printer1", printer1), ("printer2", printer2), ("printer3", printer3)):
    print(name, "|", p.print_capacity, p.color_mode, p.printer_type, "| id:", id(p))

printer1 | 500 Color Laser | id: 1467870034448
printer2 | 200 Mono Inkjet | id: 1468382187664
printer3 | 1000 Mono Dot Matrix | id: 1468382187984


## 65. MovieTicket Class

In [71]:
class MovieTicket:
    def __init__(self, movie_name, theatre, seat_number):
        self.movie_name = movie_name
        self.theatre = theatre
        self.seat_number = seat_number

ticket1 = MovieTicket("Leo", "PVR Mysuru", "A1")
ticket2 = MovieTicket("Kantara", "INOX Mysuru", "B7")
ticket3 = MovieTicket("Jawan", "DRC Cinemas", "C12")

for t in (ticket1, ticket2, ticket3):
    print(t.movie_name, "|", t.theatre, "| Seat", t.seat_number)

Leo | PVR Mysuru | Seat A1
Kantara | INOX Mysuru | Seat B7
Jawan | DRC Cinemas | Seat C12
